# Checklist Generation Pipeline

**Flow:**
1. **OCR** — Extract text from the uploaded document (any type: LC, Health Cert, B/L, etc.)
2. **Detect Document Type** — LLM identifies what kind of document it is
3. **Filter Checks** — Match checks from the verification Excel where `Check_against` (col E) contains the detected document type
4. **Output JSON Checklist** — All applicable checks saved to `final_checklist.json`

In [2]:
# Cell 1 - Install dependencies
print("Installing dependencies...")
!pip install pymupdf pillow boto3 openpyxl -q
print("Dependencies installed.")

Installing dependencies...
Dependencies installed.


'pip' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
# Cell 2 - Imports
print("Loading imports...")
import boto3, json, fitz, io, re, zipfile, os
from collections import defaultdict
import xml.etree.ElementTree as ET
from PIL import Image
print("  boto3, fitz, PIL, re, zipfile, os, ET - all loaded")
print("Imports OK")

Loading imports...
  boto3, fitz, PIL, re, zipfile, os, ET - all loaded
Imports OK


In [ ]:
# Cell 3 - Config
print("Setting config...")
AWS_ACCESS_KEY   = 
AWS_SECRET_KEY   = 
AWS_REGION       = 
BEDROCK_MODEL_ID = 

PDF_FOLDER = r"C:\Users\HITS\Desktop\New folder (2)"
VERIF_FILE = r"C:\Users\HITS\Downloads\Verification_Checks_Summary_Shared.xlsx"
OUTPUT_DIR = r"C:\Users\HITS\Desktop\New folder (2)"

print(f"  AWS Region      : {AWS_REGION}")
print(f"  Bedrock model   : {BEDROCK_MODEL_ID.split('/')[-1]}")
print(f"  PDF folder      : {PDF_FOLDER}")
print(f"  Verification XL : {VERIF_FILE}")
print(f"  Output dir      : {OUTPUT_DIR}")
print("Config ready.")

Setting config...
  AWS Region      : us-east-1
  Bedrock model   : global.amazon.nova-2-lite-v1:0
  PDF folder      : C:\Users\HITS\Desktop\New folder (2)
  Verification XL : C:\Users\HITS\Downloads\Verification_Checks_Summary_Shared.xlsx
  Output dir      : C:\Users\HITS\Desktop\New folder (2)
Config ready.


In [5]:
# Cell 4 - AWS clients
print("Initialising AWS clients...")
textract_client = boto3.client(
    "textract", region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY, aws_secret_access_key=AWS_SECRET_KEY)
print("  Textract client  : OK")

bedrock_client = boto3.client(
    "bedrock-runtime", region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY, aws_secret_access_key=AWS_SECRET_KEY)
print("  Bedrock client   : OK")
print("AWS clients ready.")

Initialising AWS clients...
  Textract client  : OK
  Bedrock client   : OK
AWS clients ready.


In [6]:
# Cell 5 - Load ALL checks from verification Excel (direct XML parse)
# Columns read:
#   A = check_number   B = category       C = check_name
#   D = description    E = checked_against  F = docs_checked
#   G = rule           H = tolerance      I = trade_type   J = phase
print("Loading verification checks from Excel...")

def read_all_checks(path):
    NS_STRICT = "http://purl.oclc.org/ooxml/spreadsheetml/main"
    NS_STD    = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
    with zipfile.ZipFile(path) as z:
        ss_xml = z.read("xl/sharedStrings.xml").decode().replace(NS_STRICT, NS_STD)
        sh_xml = z.read("xl/worksheets/sheet1.xml").decode().replace(NS_STRICT, NS_STD)
    NS = NS_STD
    strings = [si.findtext(f".//{{{NS}}}t") or "" for si in ET.fromstring(ss_xml).findall(f"{{{NS}}}si")]
    rows = []
    for row in ET.fromstring(sh_xml).findall(f".//{{{NS}}}row"):
        cells = {}
        for c in row.findall(f"{{{NS}}}c"):
            col = "".join(filter(str.isalpha, c.get("r", "")))
            v_el = c.find(f"{{{NS}}}v")
            val  = v_el.text if v_el is not None else ""
            if c.get("t", "") == "s" and val: val = strings[int(val)]
            cells[col] = val
        rows.append(cells)
    checks = []
    for r in rows[1:]:  # skip header row
        name = r.get("C", "").strip()
        if not name or r.get("I", "").strip() == "Cash": continue
        checks.append({
            "check_number":    r.get("A", "").strip(),
            "category":        r.get("B", "").strip() or "GENERAL",
            "check_name":      name,
            "description":     r.get("D", "").strip(),
            "checked_against": r.get("E", "").strip(),   # <-- KEY FILTER COLUMN
            "docs_checked":    r.get("F", "").strip(),
            "rule":            r.get("G", "").strip(),
            "tolerance":       r.get("H", "").strip(),
            "trade_type":      r.get("I", "").strip(),
            "phase":           r.get("J", "").strip(),
        })
    return checks

all_checks = read_all_checks(VERIF_FILE)

checks_by_category = defaultdict(list)
for c in all_checks:
    checks_by_category[c["category"]].append(c)

print(f"  Total checks loaded : {len(all_checks)}")
print(f"  Categories          : {len(checks_by_category)}")
print(f"  Cash-only skipped   : (excluded automatically)")
print()
print("Breakdown by category:")
for cat, items in checks_by_category.items():
    print(f"  {cat:<40} {len(items):>3} checks")
print()
print("Verification checks loaded.")

Loading verification checks from Excel...
  Total checks loaded : 114
  Categories          : 15
  Cash-only skipped   : (excluded automatically)

Breakdown by category:
  BASIC TRADE INFO                           7 checks
  PRODUCT DETAILS                           11 checks
  QUALITY SPECIFICATIONS                     9 checks
  SHIPPING DETAILS                          11 checks
  DOCUMENT REQUIREMENTS                      8 checks
  REFERENCE NUMBERS                          4 checks
  SPECIAL REQUIREMENTS                       9 checks
  COMPLIANCE STATEMENTS                      7 checks
  BANKING DETAILS                            5 checks
  CROSS-DOCUMENT VALIDATION                 10 checks
  VERSION COMPARISON                        11 checks
  UCP 600 CONTRACT CHECKS                    2 checks
  COUNTRY-SPECIFIC TEMPORAL                  3 checks
  SUPPORTING DOC VERIFICATION               13 checks
  LC AMENDMENT                               4 checks

Verification checks

In [7]:
# Cell 6 - Document-type filter
# Filters checks by matching keywords against the 'checked_against' column (col E)
# This is the column that lists which document type each check applies to.
print("Setting up document-type filter...")

# Keywords to match inside the 'checked_against' field for each document type
DOC_TYPE_KEYWORDS = {
    "Letter of Credit":                   ["LC", "Letter of Credit", "All documents", "All presented", "All uploaded", "All"],
    "Bill of Lading":                     ["Bill of Lading", "B/L"],
    "Commercial Invoice":                 ["Commercial Invoice", "Invoice"],
    "Certificate of Origin":              ["Certificate of Origin", "COO"],
    "Packing List":                       ["Packing List"],
    "Weight Certificate":                 ["Weight Certificate", "Weight Cert"],
    "Quality Certificate":                ["Quality Certificate", "Quality Cert", "Analysis Certificate"],
    "Health Certificate":                 ["Health Certificate", "Health Cert", "Health"],
    "Phytosanitary Certificate":          ["Phytosanitary"],
    "Fumigation Certificate":             ["Fumigation"],
    "Insurance Certificate":              ["Insurance Certificate", "Insurance Cert"],
    "Non-GMO Certificate":                ["Non-GMO"],
    "Non-Radiation Certificate":          ["Non-Radiation"],
    "Classification Society Certificate": ["Classification Society"],
    "P&I Insurance Certificate":          ["P&I Insurance"],
}

def filter_checks_for_doc_type(doc_type, checks):
    """
    Returns only the checks where 'checked_against' (col E) contains
    at least one keyword matching the detected document type.
    """
    kws = DOC_TYPE_KEYWORDS.get(doc_type, [])
    if not kws:
        print(f"  [FILTER] No keywords defined for doc type: {doc_type} — returning all checks")
        return checks
    matched = [
        c for c in checks
        if any(kw.lower() in c["checked_against"].lower() for kw in kws)
    ]
    return matched

print(f"  Supported document types : {len(DOC_TYPE_KEYWORDS)}")
for dt in DOC_TYPE_KEYWORDS:
    print(f"    - {dt}")
print("Doc-type filter ready.")

Setting up document-type filter...
  Supported document types : 15
    - Letter of Credit
    - Bill of Lading
    - Commercial Invoice
    - Certificate of Origin
    - Packing List
    - Weight Certificate
    - Quality Certificate
    - Health Certificate
    - Phytosanitary Certificate
    - Fumigation Certificate
    - Insurance Certificate
    - Non-GMO Certificate
    - Non-Radiation Certificate
    - Classification Society Certificate
    - P&I Insurance Certificate
Doc-type filter ready.


In [8]:
# Cell 7 - OCR + Document-type detection helpers
print("Defining OCR + document-type helpers...")

VALID_DOC_TYPES = [
    "Letter of Credit", "Bill of Lading", "Commercial Invoice",
    "Certificate of Origin", "Packing List", "Weight Certificate",
    "Quality Certificate", "Health Certificate", "Phytosanitary Certificate",
    "Fumigation Certificate", "Insurance Certificate", "Non-GMO Certificate",
    "Non-Radiation Certificate", "Classification Society Certificate",
    "P&I Insurance Certificate", "Unknown"
]

def call_llm(prompt):
    resp = bedrock_client.invoke_model(
        modelId=BEDROCK_MODEL_ID, contentType="application/json", accept="application/json",
        body=json.dumps({"messages": [{"role": "user", "content": [{"text": prompt}]}],
                         "inferenceConfig": {"maxTokens": 512, "temperature": 0}}))
    return json.loads(resp["body"].read().decode("utf-8"))["output"]["message"]["content"][0]["text"]


def pdf_to_images(pdf_path):
    doc = fitz.open(pdf_path)
    imgs = [(page.get_pixmap(matrix=fitz.Matrix(2,2)), page.number) for page in doc]
    result = [(Image.open(io.BytesIO(pix.tobytes("png"))), pg) for pix, pg in imgs]
    doc.close()
    return result


def ocr_pdf(pdf_path, max_retries=3):
    """Returns list of (page_number, text) tuples — one per page."""
    print(f"  [OCR] Starting: {os.path.basename(pdf_path)}")
    pages = pdf_to_images(pdf_path)
    print(f"  [OCR] Converted to {len(pages)} image(s)")
    page_texts = []
    for img, pg_num in pages:
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        for attempt in range(max_retries):
            try:
                resp = textract_client.analyze_document(
                    Document={"Bytes": buf.getvalue()}, FeatureTypes=["TABLES", "FORMS"])
                text = "\n".join(b.get("Text","") for b in resp.get("Blocks",[]) if b["BlockType"]=="LINE")
                page_texts.append((pg_num + 1, text))
                print(f"  [OCR] Page {pg_num+1}: {len(text)} chars extracted")
                break
            except Exception as e:
                print(f"  [OCR] Page {pg_num+1} attempt {attempt+1} failed: {e}")
                if attempt == max_retries - 1:
                    page_texts.append((pg_num+1, ""))
                    print(f"  [OCR] Page {pg_num+1}: skipped (all retries exhausted)")
    total_chars = sum(len(t) for _, t in page_texts)
    print(f"  [OCR] Complete: {len(page_texts)} pages, {total_chars} total chars")
    return page_texts


def detect_document_type(full_text):
    print("  [DOC-TYPE] Calling LLM to detect document type...")
    prompt = (
        "Identify the trade finance document type from the text below.\n"
        "Reply with ONLY one of these exact strings (nothing else):\n"
        "Letter of Credit, Bill of Lading, Commercial Invoice, Certificate of Origin,\n"
        "Packing List, Weight Certificate, Quality Certificate, Health Certificate,\n"
        "Phytosanitary Certificate, Fumigation Certificate, Insurance Certificate,\n"
        "Non-GMO Certificate, Non-Radiation Certificate,\n"
        "Classification Society Certificate, P&I Insurance Certificate, Unknown\n\n"
        "Text (first 2000 chars):\n" + full_text[:2000]
    )
    raw = call_llm(prompt).strip()
    detected = "Unknown"
    for dt in VALID_DOC_TYPES:
        if dt.lower() in raw.lower():
            detected = dt
            break
    print(f"  [DOC-TYPE] Detected: [{detected}]")
    return detected

print("  call_llm()             - defined")
print("  pdf_to_images()        - defined")
print("  ocr_pdf()              - defined")
print("  detect_document_type() - defined")
print("Helpers ready.")

Defining OCR + document-type helpers...
  call_llm()             - defined
  pdf_to_images()        - defined
  ocr_pdf()              - defined
  detect_document_type() - defined
Helpers ready.


In [9]:
# Cell 8 - OCR: extract text from all PDFs in PDF_FOLDER
# Results stored in `ocr_results` — re-run checklist cell without re-OCR-ing.

pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.lower().endswith(".pdf")]
print(f"Found {len(pdf_files)} PDF(s) in: {PDF_FOLDER}")
for f in pdf_files:
    print(f"  - {f}")
print()

ocr_results = {}   # { filename: [(page_num, text), ...] }

for pdf_file in pdf_files:
    pdf_path = os.path.join(PDF_FOLDER, pdf_file)
    print(f"[OCR] {pdf_file}")
    page_texts = ocr_pdf(pdf_path)
    ocr_results[pdf_file] = page_texts
    print()

print("=" * 55)
print("OCR COMPLETE")
print("=" * 55)
for fname, pages in ocr_results.items():
    total = sum(len(t) for _, t in pages)
    print(f"  {fname}: {len(pages)} page(s), {total} chars")

print("\n--- OCR TEXT PREVIEW ---")
for fname, pages in ocr_results.items():
    print(f"\n{'='*55}")
    print(f"FILE: {fname}")
    print(f"{'='*55}")
    for pg, txt in pages:
        print(f"\n[Page {pg}]")
        # Print only first 500 chars per page as preview
        print(txt[:500] + (" ..." if len(txt) > 500 else ""))
print("\n--- END OCR PREVIEW ---")

Found 1 PDF(s) in: C:\Users\HITS\Desktop\New folder (2)
  - LC _ N Mohammed.pdf

[OCR] LC _ N Mohammed.pdf
  [OCR] Starting: LC _ N Mohammed.pdf
  [OCR] Converted to 4 image(s)
  [OCR] Page 1: 1664 chars extracted
  [OCR] Page 2: 2896 chars extracted
  [OCR] Page 3: 3347 chars extracted
  [OCR] Page 4: 701 chars extracted
  [OCR] Complete: 4 pages, 8608 total chars

OCR COMPLETE
  LC _ N Mohammed.pdf: 4 page(s), 8608 chars

--- OCR TEXT PREVIEW ---

FILE: LC _ N Mohammed.pdf

[Page 1]
3/20/25, 5:55 PM
about:blank
NON-OPERATIVE COPY
Instance Type and Transmission
Original Received from Application - Outgoing Draft
Authorization Date
Priority/Delivery
: Normal
Message Header
Swift Input
: FIN 700 Issue of a Documentary Credit
Sender Swift address
: UCBLBDDHKTG
UNITED COMMERCIAL BANK PLC
UCBLBDDHKTG
KHATUNGANJ BRANCH
601RAMJOY MOHAJAN LANE, KHATUNGANJ
Receiver Swift address
: SCBLSG22XXX
STANDARD CHARTERED BANK (SG) LIMITED
SCBLSG22
8 MARINA BOULEVARD, ZIP CODE-018981
SINGAPORE
 ...

[Pag

In [10]:
# Cell 9 - CHECKLIST GENERATION
# For each OCR'd document:
#   STEP A: Detect document type via LLM
#   STEP B: Filter checks where checked_against (col E) matches the doc type
#   STEP C: Build JSON checklist of all applicable checks
# No LLM field extraction. No verification engine. Pure checklist.

all_results = {}

for pdf_file, page_texts in ocr_results.items():
    full_ocr = "\n".join(f"--- Page {pg} ---\n{txt}" for pg, txt in page_texts)

    print("=" * 65)
    print(f"CHECKLIST: {pdf_file}")
    print("=" * 65)

    # ── STEP A: Detect document type ──────────────────────────────────
    print("\n[STEP A] Detecting document type...")
    doc_type = detect_document_type(full_ocr)
    print(f"[STEP A] Document type: [{doc_type}]\n")

    # ── STEP B: Filter checks using checked_against (col E) ───────────
    print("[STEP B] Filtering checks via 'Check_against' column (col E)...")
    relevant_checks = filter_checks_for_doc_type(doc_type, all_checks)
    print(f"[STEP B] {len(relevant_checks)} checks apply to [{doc_type}] (out of {len(all_checks)} total)\n")

    if not relevant_checks:
        print(f"[STEP B] WARNING: No checks matched for doc type '{doc_type}'.")
        print("         Check that your verification Excel column E contains the right keywords.\n")
        all_results[pdf_file] = {
            "document_file": pdf_file,
            "document_type": doc_type,
            "total_checks": 0,
            "checks_by_category": {},
            "all_checks": []
        }
        continue

    # ── STEP C: Group by category and build checklist ─────────────────
    print("[STEP C] Building checklist...")
    cat_checks = defaultdict(list)
    for c in relevant_checks:
        cat_checks[c["category"]].append(c)

    print(f"  Categories with checks: {len(cat_checks)}")
    for cat, items in cat_checks.items():
        print(f"    {cat:<45} {len(items):>3} checks")
    print()

    # Build per-category dict — each item is a clean checklist entry
    checks_by_category_out = {}
    all_checks_flat = []

    for category, checks in cat_checks.items():
        category_items = []
        for check in checks:
            entry = {
                "check_number":    check["check_number"],
                "check_name":      check["check_name"],
                "category":        check["category"],
                "description":     check["description"],
                "checked_against": check["checked_against"],
                "docs_checked":    check["docs_checked"],
                "rule":            check["rule"],
                "tolerance":       check["tolerance"],
                "trade_type":      check["trade_type"],
                "phase":           check["phase"],
            }
            category_items.append(entry)
            all_checks_flat.append(entry)

        checks_by_category_out[category] = category_items

    all_results[pdf_file] = {
        "document_file":       pdf_file,
        "document_type":       doc_type,
        "total_checks":        len(all_checks_flat),
        "checks_by_category":  checks_by_category_out,
        "all_checks":          all_checks_flat
    }

    print(f"[DONE] {pdf_file}")
    print(f"  Document type  : {doc_type}")
    print(f"  Total checks   : {len(all_checks_flat)}")
    print(f"  Categories     : {len(checks_by_category_out)}")

print()
print("=" * 65)
print("CHECKLIST GENERATION COMPLETE")
print("=" * 65)
print(f"Processed {len(all_results)} document(s).")

CHECKLIST: LC _ N Mohammed.pdf

[STEP A] Detecting document type...
  [DOC-TYPE] Calling LLM to detect document type...
  [DOC-TYPE] Detected: [Letter of Credit]
[STEP A] Document type: [Letter of Credit]

[STEP B] Filtering checks via 'Check_against' column (col E)...
[STEP B] 82 checks apply to [Letter of Credit] (out of 114 total)

[STEP C] Building checklist...
  Categories with checks: 14
    BASIC TRADE INFO                                7 checks
    PRODUCT DETAILS                                10 checks
    QUALITY SPECIFICATIONS                          9 checks
    SHIPPING DETAILS                                9 checks
    DOCUMENT REQUIREMENTS                           8 checks
    REFERENCE NUMBERS                               4 checks
    SPECIAL REQUIREMENTS                            8 checks
    COMPLIANCE STATEMENTS                           7 checks
    BANKING DETAILS                                 3 checks
    CROSS-DOCUMENT VALIDATION                       2 

In [11]:
# Cell 10 - Print checklist grouped by category
for pdf_file, result in all_results.items():
    doc_type = result["document_type"]
    total    = result["total_checks"]

    print(f"\n{'='*70}")
    print(f"  FILE          : {pdf_file}")
    print(f"  DOCUMENT TYPE : {doc_type}")
    print(f"  TOTAL CHECKS  : {total}")
    print(f"{'='*70}")

    if total == 0:
        print("  (No checks matched for this document type)")
        continue

    for category, items in result["checks_by_category"].items():
        print(f"\n  ┌─ [{category}]  ({len(items)} checks)")
        print(f"  {'─'*67}")
        for item in items:
            num  = str(item['check_number']).rjust(4)
            name = item['check_name'][:50]
            ca   = item['checked_against'][:30]
            print(f"  │ #{num}  {name:<50}  [{ca}]")
            if item.get('rule'):
                print(f"  │        rule: {item['rule']}")
        print(f"  └{'─'*67}")


  FILE          : LC _ N Mohammed.pdf
  DOCUMENT TYPE : Letter of Credit
  TOTAL CHECKS  : 82

  ┌─ [BASIC TRADE INFO]  (7 checks)
  ───────────────────────────────────────────────────────────────────
  │ #   1  LC Number                                           [LC / DI / Contract]
  │        rule: Exact match
  │ #   2  LC Issue Date                                       [LC]
  │        rule: Valid date format; must be before expiry
  │ #   3  LC Expiry Date                                      [LC]
  │        rule: Presentation date must be on or before LC expiry
  │ #   4  Applicant (Buyer)                                   [LC]
  │        rule: 100% exact character match
  │ #   5  Beneficiary (Seller)                                [LC]
  │        rule: 100% exact character match including punctuation
  │ #  5a  Consignee Name Match (LC)                           [LC]
  │        rule: 100% exact character match including punctuation
  │ #  5b  Notify Party Match (LC)           

In [12]:
# Cell 11 - Save final_checklist.json
print("Saving checklist...")
checklist_path = os.path.join(OUTPUT_DIR, "final_checklist.json")
with open(checklist_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
size_kb = os.path.getsize(checklist_path) / 1024
print(f"  Path  : {checklist_path}")
print(f"  Size  : {size_kb:.1f} KB")
print(f"  Docs  : {len(all_results)}")
total_checks = sum(r["total_checks"] for r in all_results.values())
print(f"  Total checks across all docs: {total_checks}")
print("Saved.")

try:
    from IPython.display import FileLink, display
    display(FileLink(checklist_path, result_html_prefix="Download checklist: "))
except Exception:
    print(f"Open file manually: {checklist_path}")

Saving checklist...
  Path  : C:\Users\HITS\Desktop\New folder (2)\final_checklist.json
  Size  : 82.1 KB
  Docs  : 1
  Total checks across all docs: 82
Saved.


C:\Users\HITS\Desktop\New folder (2)\final_checklist.json

In [13]:
# Cell 12 - (Optional) Print full JSON output
print("Full JSON output:\n")
for pdf_file, result in all_results.items():
    print(f"\n=== {pdf_file} ({result['document_type']}) ===")
    # Print summary + first 3 checks per category for a quick view
    summary = {
        "document_file":  result["document_file"],
        "document_type":  result["document_type"],
        "total_checks":   result["total_checks"],
        "categories":     {
            cat: f"{len(items)} checks"
            for cat, items in result["checks_by_category"].items()
        }
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print("\n--- Sample checks (first 2 per category) ---")
    for cat, items in result["checks_by_category"].items():
        print(f"\n  [{cat}]")
        for item in items[:2]:
            print(json.dumps(item, indent=4, ensure_ascii=False))

Full JSON output:


=== LC _ N Mohammed.pdf (Letter of Credit) ===
{
  "document_file": "LC _ N Mohammed.pdf",
  "document_type": "Letter of Credit",
  "total_checks": 82,
  "categories": {
    "BASIC TRADE INFO": "7 checks",
    "PRODUCT DETAILS": "10 checks",
    "QUALITY SPECIFICATIONS": "9 checks",
    "SHIPPING DETAILS": "9 checks",
    "DOCUMENT REQUIREMENTS": "8 checks",
    "REFERENCE NUMBERS": "4 checks",
    "SPECIAL REQUIREMENTS": "8 checks",
    "COMPLIANCE STATEMENTS": "7 checks",
    "BANKING DETAILS": "3 checks",
    "CROSS-DOCUMENT VALIDATION": "2 checks",
    "VERSION COMPARISON": "6 checks",
    "COUNTRY-SPECIFIC TEMPORAL": "3 checks",
    "SUPPORTING DOC VERIFICATION": "2 checks",
    "LC AMENDMENT": "4 checks"
  }
}

--- Sample checks (first 2 per category) ---

  [BASIC TRADE INFO]
{
    "check_number": "1",
    "check_name": "LC Number",
    "category": "BASIC TRADE INFO",
    "description": "Unique identifier for the letter of credit",
    "checked_against": "LC 